In [17]:
import requests
import pandas as pd
import time
from datetime import datetime
import logging

In [18]:
SECRET_KEY = "v3.r.139738572.2f52412d5f2c2eecb33499f5208f69d65e567264.0c89e0d7bc7c38b502cd186e1d06ea81f0b5184a"

<h2>Компании Златы</h2>

In [19]:
companies = pd.read_csv('id_client_with_counts_final (2).csv')
companies.head()

,id_client,firm_name,anonymous,vacancies_count
0,11714,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,False,230
1,4961470,ПСМ,False,188
2,874575,Группа Компаний РУСАГРО,False,54
3,24172,Ростелеком,False,54
4,291920,ФГБОУ ВО Санкт-Петербургский государственный у...,False,45


In [20]:
companies.shape

(160, 4)

<h2>Вакансии</h2>

In [53]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("logging_dasha_dataset_1.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)

vacancies_list = []
vacancies_df = pd.DataFrame()
companies_id = list(companies['id_client'].unique())
url = "https://api.superjob.ru/2.0/vacancies/"
headers = {"X-Api-App-Id": SECRET_KEY}
max_page_num = 100

logging.info("Стартую")
  
vacancies_list = []
vacancies_df = pd.DataFrame()
  
max_page_num = 100
  
for i in companies_id:
    print(i)
    logging.info(f"Ищем вакансии для компании: {i}")
    companies_vacancies_num = 0

    for current_page in range(max_page_num):
        params = {
            "id_client": i,
            "count": 100,
            "page": current_page
        }
  
        logging.info(f'Компания - {i}, страница - {current_page + 1}')
  
        try:
            response = requests.get(url, headers=headers, params=params)
        except Exception as e:
            logging.error(f"Ошибка при запросе: {e}")
            continue
  
        logging.info(f"Ссылка - {url}, статус = {response.status_code}")
  
        if response.status_code != 200:
            logging.warning(f"Статус не 200: {response.status_code}, текст: {response.text}")
            continue
  
        try:
            data = response.json()
        except ValueError as e:
            logging.error(f"Проблемы с парсингом для компании {i}, страница {current_page}: {e}")
            continue
  
        for item in data.get('objects', []):  
            if item.get('id_client') != i:
                continue
  
            '''if item.get('is_archive', False) or item.get('is_storage', False):
                continue'''
  
            row = {
                'profession': item.get('profession', None),
                'id_client': item.get('id_client', None),
                'date_published': item.get('date_published', None),
                'candidat': item.get('candidat', None),
                'town': item.get('town', {}).get('title') if isinstance(item.get('town'), dict) else item.get('town'),
                'experience': item.get('experience', None),
                'catalogues': item.get('catalogues', None),
                'payment_from': item.get('payment_from', None),
                'payment_to': item.get('payment_to', None),
                'firm_name': item.get('firm_name', None),
                'firm_activity': item.get('firm_activity', None)
            }
            vacancies_list.append(row)
            companies_vacancies_num += 1
  
    logging.info(f"Собрали данные по компании - {i}. Спарсили {companies_vacancies_num} вакансий")
  

logging.info("Успешный успех")

11714
4961470
874575
24172
291920
3001379
4957734
4887815
4944790
4962080
4243646
4944476
4944244
3974341
38260
4905942
4942874
191494
4904563
4959545
4907133
3797376
4959764
4957440
4346970
272717
4859700
4952362
3440179
2631007
4918750
4451620
4954737
267938
35359
4273213
4961508
4959724
4961511
173974
4932954
2464328
298376
4763170
2722945
2416642
4070787
860291
13775
237125
4059878
139568
3595643
51445
235578
4959197
4922230
3046437
4253438
89813
196244
71817
14215
314091
2962986


ERROR:root:Ошибка при запросе: HTTPSConnectionPool(host='api.superjob.ru', port=443): Max retries exceeded with url: /2.0/vacancies/?id_client=2962986&count=100&page=26 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x13e507190>, 'Connection to api.superjob.ru timed out. (connect timeout=None)'))


2063869
335586
542741
1996534
4596048
2081428
3861304
4942854
178200
289488
14449
100593
1977505
4891649
4564603
2466656
514997
3845930
4960349
4949533
3316052
2727860
154674
4941795
4876809
4905642
3551066
3037705
692390
678972
528217
3599075
3629294
3597464
3758350
4353589
3959568
4572294
2963005
4419950
4957980
4935016
4959207
4853605
289082
267737
4894541
2985494
2720788
2230116
441150
2756555
4451188
4940742
4901219
4748161
4743716
3382148
2635306
3462139
4140409
4029196
4236166
4535808
4821130
4412294


KeyboardInterrupt: 

In [55]:
vacancies_df_current = pd.DataFrame(vacancies_list)
vacancies_df_current = vacancies_df_current.reset_index()
vacancies_df_current = vacancies_df_current.rename(columns={'index': 'vacancy_id'})
vacancies_df = pd.concat([vacancies_df, vacancies_df_current], ignore_index=True)

In [56]:
vacancies_df.shape

(6164, 12)

In [57]:
vacancies_df.head(10)

,vacancy_id,profession,id_client,date_published,candidat,town,experience,catalogues,payment_from,payment_to,firm_name,firm_activity
0,0,Дежурный специалист по информационной безопасн...,11714,1775739004,"ФФКУ ""Налог-Сервис"" ФНС России по ЦОД- это про...",Уфа,"{'id': 2, 'title': 'От 1 года'}","[{'id': 33, 'title': 'IT, Интернет, связь, тел...",0,0,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен..."
1,1,Системный администратор,11714,1775725510,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Фурманов,"{'id': 2, 'title': 'От 1 года'}","[{'id': 33, 'title': 'IT, Интернет, связь, тел...",0,36540,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен..."
2,2,Инженер по эксплуатации зданий и сооружений,11714,1775725313,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Истра,"{'id': 3, 'title': 'От 3 лет'}","[{'id': 1, 'title': 'Административная работа, ...",0,42000,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен..."
3,3,Системный администратор,11714,1775725298,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Скадовск,"{'id': 1, 'title': 'Без опыта'}","[{'id': 33, 'title': 'IT, Интернет, связь, тел...",30000,35000,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен..."
4,4,Системный администратор,11714,1775725510,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Владимир,"{'id': 1, 'title': 'Без опыта'}","[{'id': 33, 'title': 'IT, Интернет, связь, тел...",26100,0,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен..."
5,5,Специалист по обслуживанию зданий,11714,1775725903,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Санкт-Петербург,"{'id': 1, 'title': 'Без опыта'}","[{'id': 306, 'title': 'Строительство, проектир...",52200,60900,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен..."
6,6,Оператор ввода данных,11714,1775725904,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Санкт-Петербург,"{'id': 1, 'title': 'Без опыта'}","[{'id': 1, 'title': 'Административная работа, ...",34800,39150,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен..."
7,7,Системный администратор,11714,1775725902,Обязанности:\n \n• Администрирование аппаратно...,Сочи,"{'id': 2, 'title': 'От 1 года'}","[{'id': 33, 'title': 'IT, Интернет, связь, тел...",44000,59000,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен..."
8,8,Специалист по информационной безопасности,11714,1775726104,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Кемерово,"{'id': 1, 'title': 'Без опыта'}","[{'id': 33, 'title': 'IT, Интернет, связь, тел...",30000,0,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен..."
9,9,Работник архива,11714,1775726407,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Истра,"{'id': 1, 'title': 'Без опыта'}","[{'id': 1, 'title': 'Административная работа, ...",0,30450,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен..."


In [58]:
vacancies_df.to_csv('vacancies_df_all_3.csv', index=False, encoding='utf-8')

In [59]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("logging_dasha_dataset_1.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)

vacancies_list = []
vacancies_df = pd.DataFrame()
companies_id = list(companies['id_client'].unique())[130:]
url = "https://api.superjob.ru/2.0/vacancies/"
headers = {"X-Api-App-Id": SECRET_KEY}
max_page_num = 100

logging.info("Стартую")
  
vacancies_list = []
vacancies_df = pd.DataFrame()
  
max_page_num = 100
  
for i in companies_id:
    print(i)
    logging.info(f"Ищем вакансии для компании: {i}")
    companies_vacancies_num = 0

    for current_page in range(max_page_num):
        params = {
            "id_client": i,
            "count": 100,
            "page": current_page
        }
  
        logging.info(f'Компания - {i}, страница - {current_page + 1}')
  
        try:
            response = requests.get(url, headers=headers, params=params)
        except Exception as e:
            logging.error(f"Ошибка при запросе: {e}")
            continue
  
        logging.info(f"Ссылка - {url}, статус = {response.status_code}")
  
        if response.status_code != 200:
            logging.warning(f"Статус не 200: {response.status_code}, текст: {response.text}")
            continue
  
        try:
            data = response.json()
        except ValueError as e:
            logging.error(f"Проблемы с парсингом для компании {i}, страница {current_page}: {e}")
            continue
  
        for item in data.get('objects', []):  
            if item.get('id_client') != i:
                continue
  
            '''if item.get('is_archive', False) or item.get('is_storage', False):
                continue'''
  
            row = {
                'profession': item.get('profession', None),
                'id_client': item.get('id_client', None),
                'date_published': item.get('date_published', None),
                'candidat': item.get('candidat', None),
                'town': item.get('town', {}).get('title') if isinstance(item.get('town'), dict) else item.get('town'),
                'experience': item.get('experience', None),
                'catalogues': item.get('catalogues', None),
                'payment_from': item.get('payment_from', None),
                'payment_to': item.get('payment_to', None),
                'firm_name': item.get('firm_name', None),
                'firm_activity': item.get('firm_activity', None)
            }
            vacancies_list.append(row)
            companies_vacancies_num += 1
  
    logging.info(f"Собрали данные по компании - {i}. Спарсили {companies_vacancies_num} вакансий")
  

logging.info("Успешный успех")

4412294
4398800
15563
202488
190718
108428
282394
397825
400494
3538196
1975482
2182657
2531979
2914805
3776618
3107146
3512555
4580933
4502734
4484806
4745650
4450741
4918378
4809124
4940258
4942669
4953872
4957726
4958617
4960450


In [60]:
vacancies_df.to_csv('vacancies_df_all_seconde_part.csv', index=False, encoding='utf-8')